# LLM Posterior — ARC-AGI Explorer
Load tasks, inspect prompts, run LOO on a single task, and visualize results.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from dotenv import load_dotenv
load_dotenv()

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from src.data_loader import load_task, load_all_tasks
from src.grid_utils import grid_to_str, grids_match, grid_diff, ARC_COLORS
from src.hypothesis_agent import build_prompt, generate_hypothesis
from src.leave_one_out import run_task_loo
from src.evaluate import evaluate_task

TASK_DIR = pathlib.Path("data/ARC-AGI/data/training")
print(f"Tasks available: {len(list(TASK_DIR.glob('*.json')))}")

## 1. Pick a task and inspect it

In [ ]:
# Change TASK_ID to any stem from data/ARC-AGI/data/training/
TASK_ID = "007bbfb7"

task = load_task(TASK_DIR / f"{TASK_ID}.json")
print(f"Task: {task.task_id}")
print(f"Train pairs: {len(task.train)}")
print(f"Test pairs:  {len(task.test)}")
print(f"\nTrain[0] input shape:  {len(task.train[0].input)} x {len(task.train[0].input[0])}")
print(f"Train[0] output shape: {len(task.train[0].output)} x {len(task.train[0].output[0])}")

## 2. Visualize all train pairs

In [ ]:
def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = max(len(r) for r in grid) if grid else 1
    for r, row in enumerate(grid):
        for c, val in enumerate(row):
            color = ARC_COLORS.get(val, "#000000")
            rect = mpatches.FancyBboxPatch(
                (c, rows - r - 1), 1, 1,
                boxstyle="square,pad=0.02",
                facecolor=color, edgecolor="#333333", linewidth=0.8,
            )
            ax.add_patch(rect)
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=9, pad=4)


def visualize_pairs(pairs, title=""):
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(6, n * 2.5))
    if n == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=11, fontweight="bold", y=1.01)
    for i, (ax_row, pair) in enumerate(zip(axes, pairs)):
        draw_grid(ax_row[0], pair.input,  f"Pair {i+1} — Input")
        draw_grid(ax_row[1], pair.output, f"Pair {i+1} — Output")
    plt.tight_layout()
    plt.show()


visualize_pairs(task.train, title=f"{task.task_id} — Train Pairs")

## 3. Visualize test pair(s)

In [ ]:
visualize_pairs(task.test, title=f"{task.task_id} — Test Pairs")

## 4. Inspect the LLM prompt (dry run)

In [ ]:
# LOO iteration 0: hold out train[0], use train[1:] as demos
demo_pairs = task.train[1:]
test_input  = task.train[0].input

prompt = build_prompt(demo_pairs, test_input)
print(prompt)

## 5. Run LOO on this task  *(requires OPENAI_API_KEY)*

In [ ]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-4o-mini"   # change to gpt-4o for better results
llm = ChatOpenAI(model=MODEL, temperature=0)

# Jupyter supports top-level await
task_result = await run_task_loo(task, llm=llm, model=MODEL)

print(f"LOO accuracy:  {task_result.loo_accuracy:.2f}  ({sum(r.exact_match for r in task_result.loo_results)}/{len(task_result.loo_results)} exact)")
print(f"Test accuracy: {task_result.test_accuracy:.2f}  ({sum(r.exact_match for r in task_result.test_results)}/{len(task_result.test_results)} exact)")

## 6. Visualize LOO predictions vs ground truth

In [ ]:
def visualize_prediction(result, title=""):
    """Show predicted vs actual side by side, with diff overlay."""
    predicted = result.predicted_output
    actual    = result.actual_output
    diff      = grid_diff(predicted, actual)
    mismatches = set(diff["mismatches"])

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    draw_grid(axes[0], actual,    "Ground Truth")
    draw_grid(axes[1], predicted, "Predicted")

    # Diff panel: green = match, red = mismatch, grey = size issue
    ax = axes[2]
    if diff["size_match"] and predicted:
        rows = len(predicted)
        cols = len(predicted[0])
        for r in range(rows):
            for c in range(cols):
                color = "#ff4444" if (r, c) in mismatches else "#44cc44"
                rect = mpatches.FancyBboxPatch(
                    (c, rows - r - 1), 1, 1,
                    boxstyle="square,pad=0.02",
                    facecolor=color, edgecolor="white", linewidth=0.5,
                )
                ax.add_patch(rect)
        ax.set_xlim(0, cols)
        ax.set_ylim(0, rows)
    else:
        ax.text(0.5, 0.5, "Size\nMismatch", ha="center", va="center",
                fontsize=12, color="red", transform=ax.transAxes)

    match_str = "EXACT MATCH" if result.exact_match else f"cell acc: {diff['cell_accuracy']:.0%}"
    ax.set_title(f"Diff ({match_str})", fontsize=9, pad=4)
    ax.set_aspect("equal")
    ax.axis("off")

    label = f"held_out={result.held_out_index}" if not result.is_test else "test"
    fig.suptitle(f"{title} | {label} | conf={result.confidence:.2f}",
                 fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.show()


print("=== LOO predictions ===")
for r in task_result.loo_results:
    visualize_prediction(r, title=task.task_id)

In [ ]:
print("=== Test predictions ===")
for r in task_result.test_results:
    visualize_prediction(r, title=task.task_id)

## 7. Per-task metrics

In [ ]:
from src.evaluate import evaluate_task
metrics = evaluate_task(task_result)

for k, v in metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 8. Inspect a specific LOO hypothesis

In [ ]:
# Change index to inspect different LOO results
idx = 0
r = task_result.loo_results[idx]

print(f"Held out: train[{r.held_out_index}]")
print(f"Exact match: {r.exact_match}  |  Cell accuracy: {r.cell_accuracy:.2%}  |  Confidence: {r.confidence:.2f}")
print(f"\nHypothesis:\n  {r.hypothesis}")
print(f"\nReasoning:\n  {r.reasoning}")